In [1]:
import json
import random
import logging
from pathlib import Path

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Configuration ──────────────────────────────────────────────────────────────
RANDOM_SEED  = 42
OUTPUT_FILE  = "master.jsonl"

# Exact filenames (must be uploaded to /content/ in Colab)
TARGET_FILES = [
    "biology_report.jsonl",
    "chemistry_statistic.jsonl",
    "ELARA_REPORT_Algebra.jsonl",
    "ELARA_REPORT_calc.jsonl",
    "ELARA_REPORT_EN.jsonl",
    "PHYSICS_REPORT (1).jsonl",
    "arabic_report.jsonl",
]

# ── Colab base directory ───────────────────────────────────────────────────────
# In Colab, uploaded files land in /content/ by default.
# Path(__file__) does NOT work in notebooks — use the hardcoded path below.
base_dir = Path("/content")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — READ ALL FILES
# ══════════════════════════════════════════════════════════════════════════════

def load_all(base_dir: Path) -> list[dict]:
    master = []

    for filename in TARGET_FILES:
        filepath = base_dir / filename

        if not filepath.exists():
            log.warning("⚠  File not found, skipping: %s", filename)
            continue

        count   = 0
        skipped = 0

        with filepath.open(encoding="utf-8") as fh:
            for line_no, raw in enumerate(fh, start=1):
                raw = raw.strip()
                if not raw:
                    continue
                try:
                    master.append(json.loads(raw))
                    count += 1
                except json.JSONDecodeError as exc:
                    log.warning("  [%s] Line %d — bad JSON, skipped: %s", filename, line_no, exc)
                    skipped += 1

        log.info("✓  %-35s  →  %d records  |  %d skipped", filename, count, skipped)

    return master


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — SHUFFLE
# ══════════════════════════════════════════════════════════════════════════════

def shuffle(records: list[dict]) -> list[dict]:
    random.seed(RANDOM_SEED)
    random.shuffle(records)
    log.info("Shuffled %d records  (seed=%d).", len(records), RANDOM_SEED)
    return records


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SAVE
# ══════════════════════════════════════════════════════════════════════════════

def save(records: list[dict], filepath: Path) -> None:
    with filepath.open("w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    log.info("Saved  →  %s  (%d records)", filepath.name, len(records))


# ══════════════════════════════════════════════════════════════════════════════
# MAIN — runs immediately when this cell is executed
# ══════════════════════════════════════════════════════════════════════════════

log.info("=" * 55)
log.info("  Elara Merge & Shuffle  —  START")
log.info("=" * 55)

records = load_all(base_dir)

if not records:
    log.error("No records loaded. Did you upload the files to Colab?")
else:
    log.info("─" * 55)
    log.info("Total records merged: %d", len(records))
    log.info("─" * 55)

    shuffle(records)
    save(records, base_dir / OUTPUT_FILE)

    log.info("=" * 55)
    log.info("  Done! Download your file from: /content/master.jsonl")
    log.info("=" * 55)

    # ── Auto-download the output file in Colab ─────────────────────────────
    from google.colab import files
    files.download("/content/master.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>